# Cartes d'attention et diagnostic des raccourcis — notebook 11

**Objectif 4 du sujet :** « analyser les zones activées par le mécanisme d'attention pour mieux comprendre la prise de décision du modèle ».

Ce notebook répond à deux questions :

1. **Où les modèles regardent-ils ?** On compare le CNN classique et le CNN avec CBAM, avec et sans augmentation.
2. **Pourquoi le CBAM sans augmentation classe-t-il les radios assombries comme « pneumonie » ?** Deux explications sont possibles : quand l'image s'assombrit, son attention se déplace vers les bords, ou bien il lit autrement la même zone centrale. On vérifie aussi si des propriétés simples des images (luminosité, format) séparent déjà les deux classes. Un modèle peut alors apprendre ce **raccourci** au lieu des signes pulmonaires.

Seules les images d'apprentissage et de validation sont utilisées. **Le test n'est pas chargé.**

## Protocole fixé avant l'analyse

| Élément | Choix |
|---|---|
| Modèles | Les 12 modèles du notebook 10 v2 : 2 architectures × avec ou sans augmentation × graines 42, 123 et 2026. Leurs fichiers sont contrôlés par empreinte SHA-256. |
| Images | Les 953 images de validation |
| Conditions | Originale ; luminosité × 0,90 ; luminosité × 0,80. Transformations identiques aux notebooks 08 v2 et 10 v2. |
| Grad-CAM | Couche : la carte de 64 × 28 × 28 qui entre dans le classificateur (sortie du bloc 3 pour le classique, sortie du CBAM pour le CBAM). Score : logit PNEUMONIA − logit NORMAL. |
| Deux cartes par image | **Carte de décision** : zones qui soutiennent la classe prédite. **Carte pneumonie** : zones qui poussent vers PNEUMONIA, quelle que soit la prédiction. |
| Attention spatiale du CBAM | Sortie sigmoïde du module spatial (28 × 28 valeurs entre 0 et 1) |
| Régions | **Padding** : bordures noires ajoutées. **Bords** : bande de 15 % de chaque côté du rectangle de l'image. **Centre** : le reste du rectangle. Le centre est une approximation grossière du thorax, pas une segmentation des poumons. |
| Mesures | Part de la « masse » de chaque carte dans chaque région. **Enrichissement** = part de masse ÷ part de surface (1 = répartition uniforme, plus de 1 = concentration). Corrélation entre la carte d'origine et la carte de l'image assombrie. |
| Diagnostic des raccourcis | Sur l'apprentissage et la validation : format de l'image d'origine, part de padding, luminosité et contraste. Pouvoir de séparation de chaque propriété (AUC) et petits modèles logistiques entraînés sur ces seules propriétés. |

**Limites :** Grad-CAM est une approximation locale, à la résolution grossière de 28 × 28. Les cartes montrent où le modèle réagit, pas une preuve de cause. La zone « centre » ne correspond pas exactement aux poumons. La validation a déjà servi à choisir les modèles.

**Exécution :** noyau `memoire_cnn`, cellules dans l'ordre. Compter quelques minutes pour le diagnostic des images, puis environ 30 à 60 minutes pour les cartes (12 modèles × 953 images × 3 conditions). Chaque modèle terminé est sauvegardé : après une interruption, réexécuter les cellules dans l'ordre, les modèles déjà traités sont réutilisés. **Pendant l'exécution, garder OneDrive en pause et fermer les CSV du projet dans Excel.**

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
import torch.nn.functional as F
from sklearn import __version__ as version_sklearn
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
DOSSIER_08 = PROJET / "optimisation" / "robustesse_validation_v2"
DOSSIER_10 = PROJET / "optimisation" / "robustesse_augmentation_v2"
SORTIE = PROJET / "optimisation" / "cartes_attention_v1"
APPAREIL = torch.device("cpu")
LOT, TAILLE, TAILLE_CARTE = 16, 224, 28
BANDE_BORDS = 0.15
GRAINE_BRUIT = 314159
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
NOMS_CLASSES = {0: "NORMAL", 1: "PNEUMONIA"}
GRAINES = [42, 123, 2026]
CONDITIONS = [
    {"id": "originale", "type": "identite", "valeur": 0.0, "libelle": "Originale"},
    {"id": "luminosite_090", "type": "luminosite", "valeur": 0.9, "libelle": "Luminosité × 0,90"},
    {"id": "luminosite_080", "type": "luminosite", "valeur": 0.8, "libelle": "Luminosité × 0,80"},
]
# Conditions dont les prédictions sans augmentation sont enregistrées par le notebook 08 v2.
IDS_CONDITIONS_08 = {"originale", "luminosite_090", "luminosite_110", "contraste_090", "contraste_110", "bruit_001", "bruit_003"}
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
FAMILLES = [("sans", "CNN_classique"), ("avec", "CNN_classique"), ("sans", "CNN_CBAM"), ("avec", "CNN_CBAM")]
LIBELLES = {
    ("sans", "CNN_classique"): "Classique\nsans augmentation",
    ("avec", "CNN_classique"): "Classique\navec augmentation",
    ("sans", "CNN_CBAM"): "CBAM\nsans augmentation",
    ("avec", "CNN_CBAM"): "CBAM\navec augmentation",
}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def remplacer(temporaire, chemin, essais=20, attente=1.5):
    # Windows ou OneDrive peuvent verrouiller un fichier quelques secondes : on réessaie.
    for essai in range(essais):
        try:
            temporaire.replace(chemin)
            return
        except PermissionError:
            if essai == essais - 1:
                raise PermissionError(f"Impossible d'écrire {chemin} : fermer ce fichier (Excel ?) "
                                      "ou suspendre OneDrive, puis relancer la cellule.")
            time.sleep(attente)

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    remplacer(temporaire, chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    remplacer(temporaire, chemin)

def ecrire_npz(chemin, **tableaux):
    temporaire = chemin.with_suffix(".tmp")
    with open(temporaire, "wb") as fichier:
        np.savez_compressed(fichier, **tableaux)
    remplacer(temporaire, chemin)

def enregistrer_figure(figure, chemin):
    temporaire = chemin.with_suffix(".tmp")
    figure.savefig(temporaire, format="png", dpi=160, bbox_inches="tight")
    remplacer(temporaire, chemin)

for dossier in [SORTIE, SORTIE / "par_image", SORTIE / "cartes_moyennes", SORTIE / "figures"]:
    dossier.mkdir(parents=True, exist_ok=True)
print("Appareil :", APPAREIL, "| PyTorch :", torch.__version__)
print("Résultats dans :", SORTIE)

## 1. Diagnostic des raccourcis possibles dans les données

Avant de regarder les modèles, on vérifie si des propriétés simples des images séparent déjà NORMAL et PNEUMONIA :

- le **format de l'image d'origine** (largeur, hauteur, rapport largeur/hauteur) et la **part de padding**. Après redimensionnement, le format se voit dans l'épaisseur des bordures noires ;
- la **luminosité et le contraste** : moyenne, écart-type et percentiles des pixels du rectangle de l'image, sur toute l'image, au centre et sur les bords.

Pour chaque propriété, on calcule son **AUC** : 0,5 signifie qu'elle ne sépare pas du tout les classes, 1 qu'elle les sépare parfaitement. On entraîne aussi trois petits modèles logistiques sur l'apprentissage (format seul, intensité seule, les deux) et on les évalue sur la validation.

Le format ne dit rien de la maladie : s'il sépare bien les classes, c'est un **raccourci** que le CNN peut exploiter à travers les bordures. L'intensité peut contenir à la fois un vrai signal (les opacités) et des différences d'acquisition : son interprétation demande plus de prudence.

Les propriétés sont enregistrées dans `caracteristiques_images.csv` et réutilisées si le notebook est relancé.

In [ ]:
def preparer_avec_masque(image):
    gris = image.convert("L")
    preparee = ImageOps.pad(
        gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR,
        color=0, centering=(0.5, 0.5)
    )
    # Toute l'image source est marquée, même ses pixels initialement noirs.
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(
        blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST,
        color=0, centering=(0.5, 0.5)
    )
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0

def bornes_regions(masque):
    """Rectangle de l'image et zone centrale, en indices inclus."""
    lignes = np.flatnonzero(masque.any(axis=1))
    colonnes = np.flatnonzero(masque.any(axis=0))
    exiger(len(lignes) > 0 and len(colonnes) > 0, "Rectangle d'image vide.")
    r0, r1, c0, c1 = int(lignes[0]), int(lignes[-1]), int(colonnes[0]), int(colonnes[-1])
    exiger(bool(masque[r0:r1 + 1, c0:c1 + 1].all()) and int(masque.sum()) == (r1 - r0 + 1) * (c1 - c0 + 1),
           "La zone de l'image n'est pas un rectangle plein.")
    bande_h = int(round(BANDE_BORDS * (r1 - r0 + 1)))
    bande_l = int(round(BANDE_BORDS * (c1 - c0 + 1)))
    cr0, cr1, cc0, cc1 = r0 + bande_h, r1 - bande_h, c0 + bande_l, c1 - bande_l
    exiger(cr0 <= cr1 and cc0 <= cc1, "Zone centrale vide.")
    exiger(bande_h > 0 or bande_l > 0, "Zone des bords vide.")
    return {"r0": r0, "r1": r1, "c0": c0, "c1": c1, "cr0": cr0, "cr1": cr1, "cc0": cc0, "cc1": cc1}

def aires(b):
    """Part de la surface 224 × 224 occupée par chaque région."""
    rectangle = (b["r1"] - b["r0"] + 1) * (b["c1"] - b["c0"] + 1)
    centre = (b["cr1"] - b["cr0"] + 1) * (b["cc1"] - b["cc0"] + 1)
    total = TAILLE * TAILLE
    return {"aire_centre": centre / total, "aire_bords": (rectangle - centre) / total, "aire_padding": (total - rectangle) / total}

def sommes_regions(carte, b):
    """Somme d'une carte 224 × 224 dans le centre, les bords et le padding."""
    total = float(carte.sum())
    rectangle = float(carte[b["r0"]:b["r1"] + 1, b["c0"]:b["c1"] + 1].sum())
    centre = float(carte[b["cr0"]:b["cr1"] + 1, b["cc0"]:b["cc1"] + 1].sum())
    return centre, rectangle - centre, total - rectangle

def caracteristiques(chemin_relatif):
    with Image.open(IMAGES / chemin_relatif) as image:
        largeur, hauteur = image.size
        mode = image.mode
        preparee, masque = preparer_avec_masque(image)
    b = bornes_regions(masque)
    a = aires(b)
    x = preparee.astype(np.float64) / 255.0
    pixels = x[masque]
    p05, p50, p95 = np.percentile(pixels, [5, 50, 95])
    somme_centre, somme_bords, _ = sommes_regions(x, b)
    return {
        "largeur": largeur, "hauteur": hauteur, "mode_original": mode,
        "rapport_largeur_hauteur": largeur / hauteur, "fraction_padding": a["aire_padding"],
        "moyenne": float(pixels.mean()), "ecart_type": float(pixels.std()),
        "p05": float(p05), "p50": float(p50), "p95": float(p95),
        "moyenne_centre": somme_centre / (a["aire_centre"] * TAILLE * TAILLE),
        "moyenne_bords": somme_bords / (a["aire_bords"] * TAILLE * TAILLE),
    }

train_df = pd.read_csv(REPARTITION / "train.csv", dtype={"groupe": str, "empreinte": str})
val_df = pd.read_csv(REPARTITION / "val.csv", dtype={"groupe": str, "empreinte": str})
for nom, tableau, attendus in [("train", train_df, {"NORMAL": 1079, "PNEUMONIA": 2721}), ("val", val_df, {"NORMAL": 269, "PNEUMONIA": 684})]:
    exiger({"chemin_relatif", "groupe", "classe"}.issubset(tableau.columns), f"Colonnes absentes : {nom}.")
    exiger(tableau["classe"].value_counts().to_dict() == attendus, f"Effectifs inattendus : {nom}.")
    exiger(tableau["chemin_relatif"].is_unique, f"Chemins dupliqués : {nom}.")
SHA_TRAIN = empreinte_fichier(REPARTITION / "train.csv")
SHA_VAL = empreinte_fichier(REPARTITION / "val.csv")
PROTOCOLE_CARACTERISTIQUES = {
    "version": "caracteristiques_images_v1", "sha256_train_csv": SHA_TRAIN, "sha256_val_csv": SHA_VAL,
    "taille": TAILLE, "bande_bords": BANDE_BORDS, "pretraitement": "L; ImageOps.pad 224 bilineaire noir centre; /255",
}
fichier_caracteristiques = SORTIE / "caracteristiques_images.csv"
fichier_protocole_caracteristiques = SORTIE / "caracteristiques_images_protocole.json"
reutiliser = (fichier_caracteristiques.exists() and fichier_protocole_caracteristiques.exists()
              and json.loads(fichier_protocole_caracteristiques.read_text(encoding="utf-8")) == PROTOCOLE_CARACTERISTIQUES)
if reutiliser:
    CARACTERISTIQUES = pd.read_csv(fichier_caracteristiques, dtype={"groupe": str})
    print("Caractéristiques réutilisées :", len(CARACTERISTIQUES), "images.")
else:
    lignes = []
    for ensemble, tableau in [("train", train_df), ("val", val_df)]:
        for ligne in tqdm(tableau.to_dict("records"), desc=f"Caractéristiques {ensemble}"):
            lignes.append({"ensemble": ensemble, "chemin_relatif": ligne["chemin_relatif"], "groupe": ligne["groupe"],
                           "classe": ligne["classe"], **caracteristiques(ligne["chemin_relatif"])})
    CARACTERISTIQUES = pd.DataFrame(lignes)
    ecrire_csv(CARACTERISTIQUES, fichier_caracteristiques)
    ecrire_json(PROTOCOLE_CARACTERISTIQUES, fichier_protocole_caracteristiques)
    print("Caractéristiques calculées :", len(CARACTERISTIQUES), "images.")
exiger(len(CARACTERISTIQUES) == len(train_df) + len(val_df)
       and not CARACTERISTIQUES.duplicated(["ensemble", "chemin_relatif"]).any(), "Caractéristiques incomplètes.")
display(CARACTERISTIQUES.groupby(["ensemble", "classe"])["mode_original"].value_counts().rename("nombre_images").reset_index())

In [ ]:
GEOMETRIE = ["largeur", "hauteur", "rapport_largeur_hauteur", "fraction_padding"]
INTENSITE = ["moyenne", "ecart_type", "p05", "p50", "p95", "moyenne_centre", "moyenne_bords"]

def pouvoir_separateur(tableau):
    y = (tableau["classe"] == "PNEUMONIA").to_numpy(dtype=int)
    lignes = []
    for variable in GEOMETRIE + INTENSITE:
        v = tableau[variable].to_numpy(dtype=float)
        normal, pneumonie = v[y == 0], v[y == 1]
        dispersion = np.sqrt(((len(normal) - 1) * normal.var(ddof=1) + (len(pneumonie) - 1) * pneumonie.var(ddof=1)) / (len(v) - 2))
        auc = roc_auc_score(y, v) if np.ptp(v) > 0 else 0.5
        lignes.append({
            "variable": variable,
            "moyenne_NORMAL": normal.mean(), "moyenne_PNEUMONIA": pneumonie.mean(),
            "difference_standardisee": (pneumonie.mean() - normal.mean()) / dispersion if dispersion > 0 else 0.0,
            "auc_separation": max(auc, 1 - auc),
            "valeurs_plus_elevees_chez": "PNEUMONIA" if auc > 0.5 else ("NORMAL" if auc < 0.5 else "aucune"),
        })
    return pd.DataFrame(lignes)

def modele_logistique(variables):
    apprentissage = CARACTERISTIQUES[CARACTERISTIQUES["ensemble"] == "train"]
    validation = CARACTERISTIQUES[CARACTERISTIQUES["ensemble"] == "val"]
    y_apprentissage = (apprentissage["classe"] == "PNEUMONIA").to_numpy(dtype=int)
    y_validation = (validation["classe"] == "PNEUMONIA").to_numpy(dtype=int)
    modele = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000))
    modele.fit(apprentissage[variables].to_numpy(dtype=float), y_apprentissage)
    probabilites = modele.predict_proba(validation[variables].to_numpy(dtype=float))[:, 1]
    predictions = (probabilites >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_validation, predictions, labels=[0, 1]).ravel()
    return {
        "variables": " + ".join(variables),
        "auc_validation": roc_auc_score(y_validation, probabilites),
        "f1_macro_validation": f1_score(y_validation, predictions, average="macro"),
        "rappel_PNEUMONIA": tp / (tp + fn), "specificite_NORMAL": tn / (tn + fp),
    }

separation_apprentissage = pouvoir_separateur(CARACTERISTIQUES[CARACTERISTIQUES["ensemble"] == "train"])
separation_validation = pouvoir_separateur(CARACTERISTIQUES[CARACTERISTIQUES["ensemble"] == "val"])
SEPARATION = separation_apprentissage.merge(
    separation_validation[["variable", "auc_separation", "valeurs_plus_elevees_chez"]],
    on="variable", suffixes=("_apprentissage", "_validation"))
MODELES_SIMPLES = pd.DataFrame([
    {"modele": "Format seul", **modele_logistique(GEOMETRIE)},
    {"modele": "Intensité seule", **modele_logistique(INTENSITE)},
    {"modele": "Format + intensité", **modele_logistique(GEOMETRIE + INTENSITE)},
])
ecrire_csv(SEPARATION, SORTIE / "diagnostic_separation_par_variable.csv")
ecrire_csv(MODELES_SIMPLES, SORTIE / "diagnostic_modeles_simples.csv")
print("Pouvoir séparateur de chaque propriété (AUC : 0,5 = aucun, 1 = parfait). Moyennes calculées sur l'apprentissage :")
display(SEPARATION.round(4))
print("Petits modèles logistiques entraînés sur l'apprentissage, évalués sur la validation :")
display(MODELES_SIMPLES.round(4))
print("Pour comparaison, les CNN obtiennent sur la validation un F1 macro d'environ 0,95 et une AUC d'environ 0,99.")

apprentissage = CARACTERISTIQUES[CARACTERISTIQUES["ensemble"] == "train"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axe, variable, titre in zip(axes, ["moyenne", "ecart_type", "rapport_largeur_hauteur"],
                                ["Luminosité moyenne", "Contraste (écart-type des pixels)", "Rapport largeur / hauteur de l'image d'origine"]):
    for classe, couleur in [("NORMAL", "#2563eb"), ("PNEUMONIA", "#c66b14")]:
        valeurs = apprentissage.loc[apprentissage["classe"] == classe, variable]
        axe.hist(valeurs, bins=40, alpha=0.55, density=True, color=couleur, label=classe)
    axe.set_title(titre, fontsize=11)
    axe.grid(alpha=0.2)
    axe.legend(fontsize=9)
fig.suptitle("Images d'apprentissage : distribution de trois propriétés simples, par classe")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "diagnostic_distributions.png")
plt.show()
plt.close(fig)

## 2. Préparer la validation et recharger les douze modèles

Les images de validation sont préparées exactement comme dans les notebooks 08 v2 et 10 v2 : leur empreinte doit être identique. Les douze modèles sont ceux enregistrés dans le protocole du notebook 10 v2, et chaque fichier doit avoir la même empreinte SHA-256.

Pendant le calcul des cartes, chaque modèle doit aussi retrouver les scores bruts (logits) enregistrés par les notebooks 08 v2 et 10 v2, pour les trois conditions. C'est la preuve que les bons modèles et les bonnes transformations sont utilisés.

In [ ]:
def bruit_image(identifiant, forme):
    cle = f"{GRAINE_BRUIT}|{identifiant}".encode("utf-8")
    graine = int.from_bytes(hashlib.sha256(cle).digest()[:8], "little")
    generateur = np.random.Generator(np.random.PCG64(graine))
    return generateur.standard_normal(forme).astype(np.float32)

def transformer(tableau_u8, masque, condition, identifiant):
    x = tableau_u8.astype(np.float32) / 255.0
    nature, valeur = condition["type"], condition["valeur"]
    if nature == "identite":
        return x
    resultat = x.copy()
    if nature == "luminosite":
        resultat[masque] = x[masque] * np.float32(valeur)
    elif nature == "contraste":
        moyenne = x[masque].mean(dtype=np.float64)
        resultat[masque] = (x[masque] - moyenne) * valeur + moyenne
    elif nature == "bruit":
        bruit = bruit_image(identifiant, x.shape)
        resultat[masque] = x[masque] + np.float32(valeur) * bruit[masque]
    else:
        raise ValueError(f"Transformation inconnue : {nature}")
    return np.clip(resultat, 0.0, 1.0).astype(np.float32)

def fichier_reference(variante, nom, graine, condition_id):
    """Prédictions déjà enregistrées pour ce modèle et cette condition (notebook 08 v2 ou 10 v2)."""
    if variante == "sans" and condition_id in IDS_CONDITIONS_08:
        return DOSSIER_08 / "predictions" / f"{nom}_graine_{graine}__{condition_id}.csv"
    return DOSSIER_10 / "predictions" / f"{variante}__{nom}_graine_{graine}__{condition_id}.csv"

exiger((DOSSIER_10 / "termine.json").is_file(), "Le notebook 10 v2 n'est pas terminé : le lancer avant ce notebook.")
fichier_protocole_10 = DOSSIER_10 / "protocole.json"
PROTOCOLE_10 = json.loads(fichier_protocole_10.read_text(encoding="utf-8"))
exiger(PROTOCOLE_10.get("version_analyse") == "comparaison_robustesse_augmentation_v2", "Protocole du notebook 10 v2 inattendu.")
exiger(PROTOCOLE_10.get("configurations") == CONFIGURATIONS, "Configurations différentes de celles du notebook 10 v2.")
conditions_10 = {c["id"]: c for c in PROTOCOLE_10["conditions"]}
for condition in CONDITIONS:
    exiger(conditions_10.get(condition["id"]) == condition, f"Condition {condition['id']} différente de celle du notebook 10 v2.")

REELS = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
IDENTIFIANTS = val_df["chemin_relatif"].tolist()
PREPAREES = np.empty((len(val_df), TAILLE, TAILLE), dtype=np.uint8)
MASQUES = np.empty_like(PREPAREES, dtype=bool)
BORNES = []
empreinte_images = hashlib.sha256()
for i, identifiant in enumerate(tqdm(IDENTIFIANTS, desc="Préparation de la validation")):
    with Image.open(IMAGES / identifiant) as image:
        PREPAREES[i], MASQUES[i] = preparer_avec_masque(image)
    exiger(MASQUES[i].any(), "Une image ne possède aucun pixel utilisable.")
    exiger((PREPAREES[i][~MASQUES[i]] == 0).all(), "Erreur de géométrie du padding.")
    BORNES.append(bornes_regions(MASQUES[i]))
    empreinte_images.update(identifiant.encode("utf-8") + b"\0")
    empreinte_images.update(PREPAREES[i].tobytes())
    empreinte_images.update(MASQUES[i].tobytes())
SHA_IMAGES = empreinte_images.hexdigest()
exiger(SHA_VAL == PROTOCOLE_10["sha256_val_csv"], "La répartition de validation diffère de celle du notebook 10 v2.")
exiger(SHA_IMAGES == PROTOCOLE_10["sha256_images_preparees_et_masques"],
       "Les images préparées diffèrent de celles des notebooks 08 v2 et 10 v2.")
AIRES = [aires(b) for b in BORNES]
manquants = [str(fichier_reference(v, m, g, c["id"])) for v, m in FAMILLES for g in GRAINES for c in CONDITIONS
             if not fichier_reference(v, m, g, c["id"]).is_file()]
exiger(not manquants, f"{len(manquants)} fichiers de prédictions de référence manquent, par exemple : {manquants[:2]}")
print("Validation :", len(val_df), "images, identiques à celles des notebooks 08 v2 et 10 v2.")
print("Surface moyenne de chaque région dans l'image 224 × 224 : "
      f"centre {np.mean([a['aire_centre'] for a in AIRES]):.1%}, "
      f"bords {np.mean([a['aire_bords'] for a in AIRES]):.1%}, "
      f"padding {np.mean([a['aire_padding'] for a in AIRES]):.1%}.")
print("Les 36 fichiers de prédictions de référence sont présents.")

In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # v2 : connexion résiduelle, identique aux notebooks 06 v2 à 10 v2.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
PRETRAITEMENT_BASE = "L; ImageOps.pad 224 bilineaire noir centre; /255"
PRETRAITEMENTS = {"sans": PRETRAITEMENT_BASE + "; aucune augmentation", "avec": PRETRAITEMENT_BASE}

def charger_modele(variante, nom, graine, chemin):
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(protocole["architecture"] == architecture, f"Architecture différente : {nom}.")
    exiger(protocole["graine"] == graine, f"Graine incompatible : {nom}, {graine}.")
    exiger(protocole["classes"] == CLASSES and protocole["pretraitement"] == PRETRAITEMENTS[variante], "Classes ou préparation incompatibles.")
    exiger(etat["configuration"] == CONFIGURATIONS[nom], "Sauvegarde d'une autre configuration.")
    with torch.random.fork_rng(devices=[]):
        modele = classe(dropout=CONFIGURATIONS[nom]["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    exiger(sum(p.numel() for p in modele.parameters()) == nombre_parametres, "Nombre de paramètres inattendu.")
    modele.eval()
    modele.requires_grad_(False)
    return modele, int(etat["epoque"])

MODELES, INFORMATIONS = {}, []
exiger(len(PROTOCOLE_10["modeles"]) == 12, "Le protocole du notebook 10 v2 doit lister douze modèles.")
for info in PROTOCOLE_10["modeles"]:
    variante, nom, graine = info["variante"], info["modele"], int(info["graine"])
    chemin = PROJET.joinpath(*info["sauvegarde_relative"].split("/"))
    exiger(chemin.is_file(), f"Sauvegarde introuvable : {chemin}")
    exiger(empreinte_fichier(chemin) == info["sha256_sauvegarde"], f"La sauvegarde {chemin} a changé depuis le notebook 10 v2.")
    modele, epoque = charger_modele(variante, nom, graine, chemin)
    exiger(epoque == int(info["epoque"]), "Époque incompatible avec le notebook 10 v2.")
    MODELES[(variante, nom, graine)] = modele
    INFORMATIONS.append({"variante": variante, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": info["sha256_sauvegarde"], "sauvegarde_relative": info["sauvegarde_relative"]})
exiger(set(MODELES) == {(v, m, g) for v, m in FAMILLES for g in GRAINES}, "Il manque des modèles.")
if PROTOCOLE_10.get("versions") != VERSIONS:
    print("Information : versions différentes de celles du notebook 10 v2. Les scores bruts seront contrôlés pendant le calcul.")
print("Douze modèles chargés et figés.")

## 3. Calculer les cartes

Pour chaque modèle, chaque image et chaque condition :

- **Grad-CAM** : on calcule le gradient de la différence de logits (PNEUMONIA − NORMAL) par rapport à la carte de 64 × 28 × 28 qui entre dans le classificateur. La moyenne de ce gradient donne un poids à chacun des 64 canaux ; la somme pondérée des canaux donne une **carte signée**. Sa partie positive est la **carte pneumonie** (zones qui poussent vers PNEUMONIA). Sa partie négative, retournée, est la carte des zones qui poussent vers NORMAL. La **carte de décision** est celle de la classe prédite.
- **Attention spatiale** (CBAM seulement) : la carte de 28 × 28 valeurs entre 0 et 1 que le module spatial applique aux caractéristiques.

Les cartes sont agrandies à 224 × 224 par interpolation bilinéaire, puis résumées par région. Toutes les cartes ne sont pas enregistrées : seules les mesures par image et les cartes moyennes par classe le sont.

Au premier lot de chaque modèle et à la fin de chaque modèle, les logits calculés sont comparés à ceux des notebooks 08 v2 et 10 v2. En cas d'écart, le calcul s'arrête avec un message.

Un fichier par modèle est écrit dans `par_image/` et `cartes_moyennes/`. Après une interruption, les modèles terminés sont réutilisés.

In [ ]:
class Capture:
    """Conserve la sortie d'une couche pendant le passage avant du modèle."""
    def __init__(self, module):
        self.sortie = None
        self.poignee = module.register_forward_hook(self.enregistrer)
    def enregistrer(self, module, entree, sortie):
        self.sortie = sortie
    def retirer(self):
        self.poignee.remove()

def preparer_lot(indices, condition):
    x = np.stack([transformer(PREPAREES[i], MASQUES[i], condition, IDENTIFIANTS[i]) for i in indices])
    return torch.from_numpy(x).unsqueeze(1)

def calculer_lot(nom, modele, x):
    """Logits (B, 2), carte Grad-CAM signée (B, 28, 28) et attention spatiale (B, 28, 28) ou None."""
    couche = modele.cbam if nom == "CNN_CBAM" else modele.bloc3
    capture = Capture(couche)
    capture_spatiale = Capture(modele.cbam.attention_spatiale.convolution) if nom == "CNN_CBAM" else None
    try:
        entree = x.clone().requires_grad_(True)
        with torch.enable_grad():
            logits = modele(entree)
            activation = capture.sortie
            exiger(activation is not None and tuple(activation.shape) == (x.shape[0], 64, TAILLE_CARTE, TAILLE_CARTE),
                   "La couche utilisée pour Grad-CAM n'a pas la forme attendue.")
            difference = logits[:, 1] - logits[:, 0]
            (gradient,) = torch.autograd.grad(difference.sum(), activation)
        poids = gradient.mean(dim=(2, 3), keepdim=True)
        carte_signee = (poids * activation.detach()).sum(dim=1)
        spatiale = None
        if capture_spatiale is not None:
            sortie = capture_spatiale.sortie
            exiger(sortie is not None and tuple(sortie.shape) == (x.shape[0], 1, TAILLE_CARTE, TAILLE_CARTE),
                   "L'attention spatiale n'a pas la forme attendue.")
            spatiale = torch.sigmoid(sortie.detach())[:, 0].numpy()
    finally:
        capture.retirer()
        if capture_spatiale is not None:
            capture_spatiale.retirer()
    logits = logits.detach().numpy()
    carte_signee = carte_signee.numpy()
    exiger(np.isfinite(logits).all() and np.isfinite(carte_signee).all(), "Valeurs non finies dans les logits ou la carte.")
    return logits, carte_signee, spatiale

def agrandir(cartes):
    """Agrandit des cartes (B, 28, 28) en (B, 224, 224) par interpolation bilinéaire."""
    t = torch.from_numpy(np.ascontiguousarray(cartes, dtype=np.float32)).unsqueeze(1)
    return F.interpolate(t, size=(TAILLE, TAILLE), mode="bilinear", align_corners=False)[:, 0].numpy().astype(np.float64)

def mesures_carte(carte, b):
    """Parts de la masse d'une carte positive dans le centre, les bords et le padding."""
    centre, bords, padding = sommes_regions(carte, b)
    total = centre + bords + padding
    if not np.isfinite(total) or total <= 1e-12:
        return {"centre": np.nan, "bords": np.nan, "padding": np.nan, "nulle": True}
    return {"centre": centre / total, "bords": bords / total, "padding": padding / total, "nulle": False}

def moyennes_attention(carte, b, a):
    """Valeur moyenne de l'attention spatiale dans chaque région."""
    centre, bords, padding = sommes_regions(carte, b)
    n = TAILLE * TAILLE
    return {"attention_centre": centre / (a["aire_centre"] * n),
            "attention_bords": bords / (a["aire_bords"] * n) if a["aire_bords"] > 0 else np.nan,
            "attention_padding": padding / (a["aire_padding"] * n) if a["aire_padding"] > 0 else np.nan}

def correlation(carte_a, carte_b):
    a = np.asarray(carte_a, dtype=np.float64).ravel()
    b = np.asarray(carte_b, dtype=np.float64).ravel()
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def cumuler(sommes, cle, carte):
    if cle not in sommes:
        sommes[cle] = [np.zeros((TAILLE, TAILLE), dtype=np.float64), 0]
    sommes[cle][0] += carte
    sommes[cle][1] += 1

def cle_modele(variante, nom, graine):
    return f"{variante}__{nom}_graine_{graine}"

def controler_logits(logits_calcules, reference, indices, contexte):
    attendus = reference[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)[indices]
    ecart = float(np.abs(logits_calcules[indices].astype(np.float64) - attendus).max())
    tolerance = 1e-4 + 1e-5 * float(np.abs(attendus).max())
    exiger(ecart <= tolerance,
           f"{contexte} : scores bruts différents des prédictions de référence (écart maximal {ecart:.2e}). "
           "Vérifier le noyau, les versions et les sauvegardes avant de continuer.")
    return ecart

def traiter_modele(variante, nom, graine):
    cle = cle_modele(variante, nom, graine)
    fichier_csv = SORTIE / "par_image" / f"{cle}.csv"
    fichier_npz = SORTIE / "cartes_moyennes" / f"{cle}.npz"
    if fichier_csv.exists() and fichier_npz.exists():
        tableau = pd.read_csv(fichier_csv, dtype={"groupe": str})
        exiger(len(tableau) == len(val_df) * len(CONDITIONS) and (tableau["id_protocole"] == ID_PROTOCOLE).all(),
               f"{cle} : résultats existants incompatibles avec ce protocole. Choisir un autre dossier SORTIE.")
        return tableau, True
    modele = MODELES[(variante, nom, graine)]
    references = {}
    for condition in CONDITIONS:
        reference = pd.read_csv(fichier_reference(variante, nom, graine, condition["id"]), dtype={"groupe": str})
        exiger(reference["chemin_relatif"].tolist() == IDENTIFIANTS,
               f"{cle} : ordre des images différent dans la référence {condition['id']}.")
        references[condition["id"]] = reference
    n = len(val_df)
    logits = {c["id"]: np.empty((n, 2), dtype=np.float32) for c in CONDITIONS}
    signees = {c["id"]: np.empty((n, TAILLE_CARTE, TAILLE_CARTE), dtype=np.float32) for c in CONDITIONS}
    sommes, lignes = {}, []
    for debut in tqdm(range(0, n, LOT), desc=cle, leave=False):
        indices = np.arange(debut, min(debut + LOT, n))
        for condition in CONDITIONS:
            cid = condition["id"]
            lg, signee, spatiale = calculer_lot(nom, modele, preparer_lot(indices, condition))
            logits[cid][indices] = lg
            signees[cid][indices] = signee
            if debut == 0:
                controler_logits(logits[cid], references[cid], indices, f"{cle}, {cid}, premier lot")
            pneumonie = agrandir(np.maximum(signee, 0.0))
            normale = agrandir(np.maximum(-signee, 0.0))
            attention = agrandir(spatiale) if spatiale is not None else None
            for k, i in enumerate(indices):
                b, a = BORNES[i], AIRES[i]
                prediction = int(lg[k, 1] > lg[k, 0])
                decision = pneumonie[k] if prediction == 1 else normale[k]
                m_decision = mesures_carte(decision, b)
                m_pneumonie = mesures_carte(pneumonie[k], b)
                ligne = {
                    "variante": variante, "modele": nom, "graine": graine, "condition": cid,
                    "chemin_relatif": IDENTIFIANTS[i], "groupe": val_df["groupe"].iat[i], "classe_vraie": int(REELS[i]),
                    "prediction": prediction, "difference_logits": float(lg[k, 1]) - float(lg[k, 0]),
                    "correcte": prediction == int(REELS[i]), **a,
                    "masse_centre_decision": m_decision["centre"], "masse_bords_decision": m_decision["bords"],
                    "masse_padding_decision": m_decision["padding"], "carte_decision_nulle": m_decision["nulle"],
                    "masse_centre_pneumonie": m_pneumonie["centre"], "masse_bords_pneumonie": m_pneumonie["bords"],
                    "masse_padding_pneumonie": m_pneumonie["padding"], "carte_pneumonie_nulle": m_pneumonie["nulle"],
                }
                if attention is not None:
                    ligne.update(moyennes_attention(attention[k], b, a))
                    cumuler(sommes, f"attention__{cid}__{int(REELS[i])}", attention[k])
                else:
                    ligne.update({"attention_centre": np.nan, "attention_bords": np.nan, "attention_padding": np.nan})
                if not m_decision["nulle"]:
                    cumuler(sommes, f"decision__{cid}__{int(REELS[i])}", decision / decision.max())
                lignes.append(ligne)
    tableau = pd.DataFrame(lignes)
    ecarts = {}
    for condition in CONDITIONS:
        cid = condition["id"]
        ecarts[cid] = controler_logits(logits[cid], references[cid], np.arange(n), f"{cle}, {cid}")
        selection = tableau["condition"] == cid
        calculees = tableau.loc[selection, "prediction"].to_numpy()
        attendues = references[cid]["prediction"].to_numpy()
        differentes = calculees != attendues
        if differentes.any():
            quasi_egalites = np.abs(tableau.loc[selection, "difference_logits"].to_numpy()[differentes]) < 1e-3
            exiger(quasi_egalites.all(), f"{cle}, {cid} : {int(differentes.sum())} prédictions diffèrent de la référence.")
            print(f"Information : {cle}, {cid} : {int(differentes.sum())} prédiction(s) quasi à égalité diffèrent de la référence.")
    index_image = {chemin: i for i, chemin in enumerate(IDENTIFIANTS)}
    predictions_originales = dict(zip(IDENTIFIANTS, logits["originale"].argmax(axis=1)))
    correlations = {cid: np.array([np.nan] * n) if cid == "originale" else
                    np.array([correlation(signees["originale"][i], signees[cid][i]) for i in range(n)])
                    for cid in signees}
    tableau["change_vs_originale"] = tableau["prediction"].to_numpy() != tableau["chemin_relatif"].map(predictions_originales).to_numpy()
    tableau["correlation_carte_signee"] = [correlations[c][index_image[p]] for c, p in zip(tableau["condition"], tableau["chemin_relatif"])]
    tableau["ecart_max_logits_reference"] = tableau["condition"].map(ecarts)
    tableau["id_protocole"] = ID_PROTOCOLE
    ecrire_npz(fichier_npz, **{f"{cle_somme}__somme": valeurs[0].astype(np.float32) for cle_somme, valeurs in sommes.items()},
               **{f"{cle_somme}__nombre": np.array(valeurs[1]) for cle_somme, valeurs in sommes.items()})
    ecrire_csv(tableau, fichier_csv)
    return tableau, False

print("Fonctions prêtes. La cellule suivante calcule les cartes.")

In [ ]:
PROTOCOLE = {
    "version_analyse": "cartes_attention_v1", "ensemble": "val", "effectif": len(val_df),
    "conditions": CONDITIONS, "lot": LOT, "taille": TAILLE, "bande_bords": BANDE_BORDS,
    "couche_gradcam": {"CNN_classique": "sortie de bloc3 (64x28x28)", "CNN_CBAM": "sortie du module CBAM residuel (64x28x28)"},
    "score_gradcam": "logit PNEUMONIA - logit NORMAL; poids = moyenne spatiale du gradient; carte signee = somme ponderee des activations",
    "cartes": {"pneumonie": "ReLU(carte signee)", "normale": "ReLU(-carte signee)",
               "decision": "carte de la classe predite (argmax des logits, NORMAL en cas d egalite)"},
    "agrandissement": "ReLU a 28x28 puis interpolation bilineaire vers 224x224, align_corners=False",
    "attention_spatiale": "sigmoide de la convolution du module spatial du CBAM (28x28)",
    "regions": "padding ajoute; bords = bande de 15% de chaque cote du rectangle image; centre = reste du rectangle",
    "correlation": "Pearson entre cartes signees 28x28, condition perturbee contre originale",
    "cartes_moyennes": "cartes de decision normalisees a un maximum de 1 par image; attention brute; moyennes par classe vraie",
    "sha256_protocole_10": empreinte_fichier(fichier_protocole_10),
    "sha256_val_csv": SHA_VAL, "sha256_images_preparees_et_masques": SHA_IMAGES,
    "modeles": INFORMATIONS, "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "limite": "Validation deja utilisee pour la selection; Grad-CAM approximation locale a 28x28; centre = approximation du thorax, pas une segmentation des poumons; aucune donnee de test.",
}
ID_PROTOCOLE = hashlib.sha256(json.dumps(PROTOCOLE, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
           "Le protocole ou les fichiers ont changé. Conserver le dossier existant et choisir un autre nom SORTIE.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)

tableaux = []
for variante, nom in FAMILLES:
    for graine in GRAINES:
        debut = time.perf_counter()
        tableau, reutilise = traiter_modele(variante, nom, graine)
        tableaux.append(tableau)
        etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut) / 60:.1f} min"
        ecart = tableau["ecart_max_logits_reference"].max()
        print(f"{variante:>4} | {nom:<13} | graine {graine:>4} | écart maximal aux logits de référence {ecart:.1e} | {etat}")
PAR_IMAGE = pd.concat(tableaux, ignore_index=True)
exiger(len(PAR_IMAGE) == 12 * len(val_df) * len(CONDITIONS), "Résultats incomplets.")
ecrire_csv(PAR_IMAGE, SORTIE / "cartes_par_image.csv")
print("Cartes calculées pour les douze modèles :", len(PAR_IMAGE), "lignes.")

## 4. Résumer : où les modèles regardent-ils ?

Les moyennes sont d'abord calculées pour chaque graine, puis résumées sur les trois graines (moyenne et écart-type). Comment lire les tableaux :

- **part de masse au centre, sur les bords, dans le padding** : où se trouve la carte de décision ;
- **enrichissement du centre** : supérieur à 1 si la carte est plus concentrée au centre qu'une carte uniforme ;
- **attention spatiale** (CBAM) : valeur moyenne, entre 0 et 1, de l'attention au centre et sur les bords. Un écart centre − bords positif signifie que l'attention favorise le centre ;
- **perturbations** : pour chaque image assombrie, corrélation entre sa carte signée et celle de l'image d'origine (1 = même carte), et variation de la part de la carte pneumonie au centre et sur les bords. Les images sont séparées selon que la prédiction reste stable ou change.

In [ ]:
T = PAR_IMAGE.copy()
T["enrichissement_centre_decision"] = T["masse_centre_decision"] / T["aire_centre"]
T["enrichissement_bords_decision"] = T["masse_bords_decision"] / T["aire_bords"]
T["ecart_attention_centre_bords"] = T["attention_centre"] - T["attention_bords"]

MESURES_DECISION = ["masse_centre_decision", "masse_bords_decision", "masse_padding_decision",
                    "enrichissement_centre_decision", "enrichissement_bords_decision", "carte_decision_nulle"]
MESURES_ATTENTION = ["attention_centre", "attention_bords", "attention_padding", "ecart_attention_centre_bords"]

def resumer(tableau, groupes, mesures):
    """Moyenne par graine, puis moyenne et écart-type sur les graines ; nombre moyen d'images par graine."""
    par_graine = tableau.groupby(groupes + ["graine"], sort=False)[mesures].mean().reset_index()
    resume = par_graine.groupby(groupes, sort=False)[mesures].agg(["mean", "std"])
    resume.columns = [f"{mesure}_{statistique}" for mesure, statistique in resume.columns]
    effectifs = (tableau.groupby(groupes + ["graine"], sort=False).size().rename("n").reset_index()
                 .groupby(groupes, sort=False)["n"].mean().rename("images_par_graine"))
    return resume.join(effectifs).reset_index()

originale = T[T["condition"] == "originale"]
SYNTHESE_ORIGINALE = resumer(originale, ["variante", "modele"], MESURES_DECISION + MESURES_ATTENTION)
SYNTHESE_CLASSES = resumer(originale, ["variante", "modele", "classe_vraie", "correcte"],
                           ["masse_centre_decision", "masse_bords_decision", "masse_padding_decision", "enrichissement_centre_decision"])
SYNTHESE_CONDITIONS = resumer(T, ["variante", "modele", "condition"],
                              ["masse_centre_decision", "masse_bords_decision", "masse_padding_decision", "enrichissement_centre_decision",
                               "attention_centre", "attention_bords", "ecart_attention_centre_bords"])

cles_image = ["variante", "modele", "graine", "chemin_relatif"]
base = originale[cles_image + ["prediction", "masse_centre_pneumonie", "masse_bords_pneumonie", "masse_padding_pneumonie"]]
PERTURBEES = T[T["condition"] != "originale"].merge(base, on=cles_image, suffixes=("", "_originale"), validate="many_to_one")
for region in ["centre", "bords", "padding"]:
    PERTURBEES[f"variation_masse_{region}_pneumonie"] = (PERTURBEES[f"masse_{region}_pneumonie"]
                                                        - PERTURBEES[f"masse_{region}_pneumonie_originale"])
PERTURBEES["type_changement"] = np.select(
    [PERTURBEES["prediction"] == PERTURBEES["prediction_originale"],
     (PERTURBEES["prediction_originale"] == 0) & (PERTURBEES["prediction"] == 1)],
    ["stable", "NORMAL → PNEUMONIA"], default="PNEUMONIA → NORMAL")
SYNTHESE_PERTURBATIONS = resumer(PERTURBEES, ["variante", "modele", "condition", "type_changement"],
                                 ["correlation_carte_signee", "variation_masse_centre_pneumonie",
                                  "variation_masse_bords_pneumonie", "variation_masse_padding_pneumonie"])

par_graine = originale.groupby(["variante", "modele", "graine"])[["enrichissement_centre_decision", "masse_bords_decision",
                                                                  "masse_padding_decision"]].mean().reset_index()
lignes = []
for mesure in ["enrichissement_centre_decision", "masse_bords_decision", "masse_padding_decision"]:
    pivot = par_graine.pivot_table(index=["variante", "graine"], columns="modele", values=mesure)
    ecarts = (pivot["CNN_CBAM"] - pivot["CNN_classique"]).rename("ecart").reset_index()
    for variante, groupe in ecarts.groupby("variante", sort=False):
        lignes.append({"mesure": mesure, "variante": variante,
                       "ecart_CBAM_moins_classique_moyen": groupe["ecart"].mean(),
                       "ecart_minimum": groupe["ecart"].min(), "ecart_maximum": groupe["ecart"].max(),
                       "graines_ou_CBAM_superieur": int((groupe["ecart"] > 0).sum())})
COMPARAISON_ARCHITECTURES = pd.DataFrame(lignes)

ecrire_csv(SYNTHESE_ORIGINALE, SORTIE / "synthese_cartes_originale.csv")
ecrire_csv(SYNTHESE_CLASSES, SORTIE / "synthese_cartes_par_classe.csv")
ecrire_csv(SYNTHESE_CONDITIONS, SORTIE / "synthese_cartes_par_condition.csv")
ecrire_csv(SYNTHESE_PERTURBATIONS, SORTIE / "synthese_perturbations.csv")
ecrire_csv(COMPARAISON_ARCHITECTURES, SORTIE / "comparaison_architectures_cartes.csv")

print("Images d'origine — où se trouve la carte de décision (moyenne et écart-type sur 3 graines) :")
display(SYNTHESE_ORIGINALE[["variante", "modele", "masse_centre_decision_mean", "masse_bords_decision_mean",
                            "masse_padding_decision_mean", "enrichissement_centre_decision_mean",
                            "enrichissement_centre_decision_std", "carte_decision_nulle_mean"]].round(4))
print("Attention spatiale du CBAM sur les images d'origine :")
display(SYNTHESE_ORIGINALE[SYNTHESE_ORIGINALE["modele"] == "CNN_CBAM"][["variante", "attention_centre_mean", "attention_bords_mean",
                                                                        "attention_padding_mean", "ecart_attention_centre_bords_mean"]].round(4))
print("Écart CBAM − classique, apparié par graine :")
display(COMPARAISON_ARCHITECTURES.round(4))
print("Images assombries — stabilité des cartes selon que la prédiction change ou non :")
display(SYNTHESE_PERTURBATIONS[["variante", "modele", "condition", "type_changement", "images_par_graine",
                                "correlation_carte_signee_mean", "variation_masse_centre_pneumonie_mean",
                                "variation_masse_bords_pneumonie_mean", "variation_masse_padding_pneumonie_mean"]].round(4))

In [ ]:
def cartes_moyennes_modele(variante, nom, graine):
    with np.load(SORTIE / "cartes_moyennes" / f"{cle_modele(variante, nom, graine)}.npz") as donnees:
        return {cle: donnees[cle] for cle in donnees.files}

def carte_moyenne_famille(variante, nom, type_carte, condition_id, classe):
    """Moyenne sur les graines des cartes moyennes par classe vraie."""
    cartes = []
    for graine in GRAINES:
        donnees = cartes_moyennes_modele(variante, nom, graine)
        cle = f"{type_carte}__{condition_id}__{classe}"
        if f"{cle}__somme" in donnees and int(donnees[f"{cle}__nombre"]) > 0:
            cartes.append(donnees[f"{cle}__somme"].astype(np.float64) / int(donnees[f"{cle}__nombre"]))
    return np.mean(cartes, axis=0) if cartes else None

FOND = {classe: PREPAREES[REELS == classe].mean(axis=0) / 255.0 for classe in [0, 1]}

# Figure 1 : carte de décision moyenne, images d'origine.
fig, axes = plt.subplots(2, 4, figsize=(16, 8.6))
for ligne, classe in enumerate([0, 1]):
    for colonne, famille in enumerate(FAMILLES):
        axe = axes[ligne, colonne]
        axe.imshow(FOND[classe], cmap="gray", vmin=0, vmax=1)
        carte = carte_moyenne_famille(*famille, "decision", "originale", classe)
        if carte is not None:
            axe.imshow(carte / carte.max(), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axe.set_title(f"{LIBELLES[famille]}\nradios {NOMS_CLASSES[classe]}", fontsize=10)
        axe.axis("off")
fig.suptitle("Carte de décision Grad-CAM moyenne sur la validation (images d'origine, 3 graines)\n"
             "Fond : image moyenne de la classe. Rouge : zones les plus utilisées pour la décision.")
fig.tight_layout(rect=(0, 0, 1, 0.95))
enregistrer_figure(fig, SORTIE / "figures" / "cartes_decision_moyennes.png")
plt.show()
plt.close(fig)

# Figure 2 : attention spatiale moyenne du CBAM.
cartes_attention = {(variante, classe): carte_moyenne_famille(variante, "CNN_CBAM", "attention", "originale", classe)
                    for variante in ["sans", "avec"] for classe in [0, 1]}
valeurs = np.concatenate([c.ravel() for c in cartes_attention.values() if c is not None])
fig, axes = plt.subplots(2, 2, figsize=(9.5, 9))
for ligne, classe in enumerate([0, 1]):
    for colonne, variante in enumerate(["sans", "avec"]):
        axe = axes[ligne, colonne]
        axe.imshow(FOND[classe], cmap="gray", vmin=0, vmax=1)
        image = axe.imshow(cartes_attention[(variante, classe)], cmap="viridis", alpha=0.55,
                           vmin=float(valeurs.min()), vmax=float(valeurs.max()))
        axe.set_title(f"{LIBELLES[(variante, 'CNN_CBAM')]}\nradios {NOMS_CLASSES[classe]}", fontsize=10)
        axe.axis("off")
fig.colorbar(image, ax=axes, fraction=0.03, label="attention spatiale moyenne (0 à 1)")
fig.suptitle("Attention spatiale moyenne du CBAM (images d'origine, 3 graines)")
enregistrer_figure(fig, SORTIE / "figures" / "attention_spatiale_moyenne.png")
plt.show()
plt.close(fig)

# Figure 3 : mesures par région et stabilité sous assombrissement.
ordre_conditions = [c["id"] for c in CONDITIONS]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))
positions = np.arange(len(FAMILLES))
resume_origine = SYNTHESE_ORIGINALE.set_index(["variante", "modele"]).loc[FAMILLES]
bas = np.zeros(len(FAMILLES))
for region, couleur in [("centre", "#15803d"), ("bords", "#c66b14"), ("padding", "#6b7280")]:
    hauteurs = resume_origine[f"masse_{region}_decision_mean"].to_numpy()
    axes[0].bar(positions, hauteurs, bottom=bas, color=couleur, label=region)
    bas += hauteurs
axes[0].set_xticks(positions, [LIBELLES[f] for f in FAMILLES], fontsize=8)
axes[0].set_ylabel("Part de la carte de décision")
axes[0].set_title("Où se trouve la carte de décision (images d'origine)")
axes[0].legend(fontsize=9)
for decalage, famille, couleur in zip([-0.15, -0.05, 0.05, 0.15], FAMILLES, ["#93c5fd", "#1d4ed8", "#fdba74", "#c2410c"]):
    tableau = SYNTHESE_CONDITIONS.set_index(["variante", "modele", "condition"]).loc[[(*famille, c) for c in ordre_conditions]]
    axes[1].errorbar(np.arange(len(ordre_conditions)) + decalage, tableau["enrichissement_centre_decision_mean"],
                     yerr=tableau["enrichissement_centre_decision_std"], fmt="o", capsize=3, color=couleur,
                     label=LIBELLES[famille].replace("\n", " "))
axes[1].axhline(1, color="gray", linewidth=1, linestyle="--")
axes[1].set_xticks(np.arange(len(ordre_conditions)), [c["libelle"] for c in CONDITIONS], fontsize=9)
axes[1].set_ylabel("Enrichissement du centre (1 = uniforme)")
axes[1].set_title("Concentration de la carte de décision au centre")
axes[1].legend(fontsize=8)
largeur = 0.38
sombre = SYNTHESE_PERTURBATIONS[SYNTHESE_PERTURBATIONS["condition"] == "luminosite_080"].set_index(["variante", "modele", "type_changement"])
for decalage, type_changement, couleur in [(-largeur / 2, "stable", "#6b7280"), (largeur / 2, "NORMAL → PNEUMONIA", "#dc2626")]:
    moyennes = [sombre["correlation_carte_signee_mean"].get((*f, type_changement), np.nan) for f in FAMILLES]
    axes[2].bar(positions + decalage, moyennes, width=largeur, color=couleur, label=f"prédiction {type_changement}")
axes[2].set_xticks(positions, [LIBELLES[f] for f in FAMILLES], fontsize=8)
axes[2].set_ylim(-1, 1)
axes[2].axhline(0, color="gray", linewidth=1)
axes[2].set_ylabel("Corrélation avec la carte d'origine")
axes[2].set_title("Luminosité × 0,80 : la carte reste-t-elle la même ?")
axes[2].legend(fontsize=9)
for axe in axes:
    axe.grid(axis="y", alpha=0.2)
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "mesures_regions.png")
plt.show()
plt.close(fig)

In [ ]:
def cartes_images(variante, nom, condition, indices, type_carte):
    """Cartes 224 × 224 de quelques images, moyennées sur les trois graines."""
    cumul = np.zeros((len(indices), TAILLE, TAILLE))
    for graine in GRAINES:
        lg, signee, spatiale = calculer_lot(nom, MODELES[(variante, nom, graine)], preparer_lot(indices, condition))
        if type_carte == "attention":
            cumul += agrandir(spatiale)
            continue
        if type_carte == "decision":
            pneumonie_predite = (lg[:, 1] > lg[:, 0])[:, None, None]
            choisies = np.where(pneumonie_predite, np.maximum(signee, 0.0), np.maximum(-signee, 0.0))
        else:
            choisies = np.maximum(signee, 0.0)
        cartes = agrandir(choisies)
        maxima = cartes.reshape(len(indices), -1).max(axis=1)[:, None, None]
        cumul += np.divide(cartes, maxima, out=np.zeros_like(cartes), where=maxima > 0)
    return cumul / len(GRAINES)

def paires_pneumonie(variante, nom, indices, condition_a, condition_b):
    """Cartes pneumonie de deux conditions, sur une échelle commune par image et par graine, puis moyennées."""
    cumul_a = np.zeros((len(indices), TAILLE, TAILLE))
    cumul_b = np.zeros((len(indices), TAILLE, TAILLE))
    for graine in GRAINES:
        modele = MODELES[(variante, nom, graine)]
        carte_a = agrandir(np.maximum(calculer_lot(nom, modele, preparer_lot(indices, condition_a))[1], 0.0))
        carte_b = agrandir(np.maximum(calculer_lot(nom, modele, preparer_lot(indices, condition_b))[1], 0.0))
        maxima = np.maximum(carte_a.reshape(len(indices), -1).max(axis=1), carte_b.reshape(len(indices), -1).max(axis=1))[:, None, None]
        cumul_a += np.divide(carte_a, maxima, out=np.zeros_like(carte_a), where=maxima > 0)
        cumul_b += np.divide(carte_b, maxima, out=np.zeros_like(carte_b), where=maxima > 0)
    return cumul_a / len(GRAINES), cumul_b / len(GRAINES)

def afficher(axe, image, carte=None, attention=False):
    axe.imshow(image, cmap="gray", vmin=0, vmax=1)
    if carte is not None:
        if attention:
            axe.imshow(carte, cmap="viridis", alpha=0.55, vmin=float(carte.min()), vmax=float(carte.max()))
        else:
            axe.imshow(carte, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    axe.set_xticks([])
    axe.set_yticks([])

# Figure 4 : exemples fixés à l'avance (les trois premières images de chaque classe dans la validation).
exemples = np.array(list(np.flatnonzero(REELS == 0)[:3]) + list(np.flatnonzero(REELS == 1)[:3]))
originale_condition = CONDITIONS[0]
colonnes = [(famille, "decision") for famille in FAMILLES] + [(("sans", "CNN_CBAM"), "attention"), (("avec", "CNN_CBAM"), "attention")]
cartes = {colonne: cartes_images(*colonne[0], originale_condition, exemples, colonne[1]) for colonne in colonnes}
fig, axes = plt.subplots(len(exemples), len(colonnes) + 1, figsize=(2.5 * (len(colonnes) + 1), 2.6 * len(exemples)))
for ligne, i in enumerate(exemples):
    image = PREPAREES[i] / 255.0
    afficher(axes[ligne, 0], image)
    axes[ligne, 0].set_ylabel(f"{NOMS_CLASSES[int(REELS[i])]}\n{Path(IDENTIFIANTS[i]).name[:22]}", fontsize=8)
    for colonne, (famille, type_carte) in enumerate(colonnes, start=1):
        afficher(axes[ligne, colonne], image, cartes[(famille, type_carte)][ligne], attention=(type_carte == "attention"))
        if ligne == 0:
            titre = LIBELLES[famille] + ("\n(attention spatiale)" if type_carte == "attention" else "\n(Grad-CAM décision)")
            axes[ligne, colonne].set_title(titre, fontsize=8)
axes[0, 0].set_title("Image", fontsize=8)
fig.suptitle("Exemples fixés à l'avance : trois premières radios NORMAL et PNEUMONIA de la validation (cartes moyennées sur 3 graines)")
fig.tight_layout(rect=(0, 0, 1, 0.97))
enregistrer_figure(fig, SORTIE / "figures" / "exemples_cartes.png")
plt.show()
plt.close(fig)

# Figure 5 : radios normales que le CBAM sans augmentation classe PNEUMONIA une fois assombries (× 0,80).
cbam_sans = PAR_IMAGE[(PAR_IMAGE["variante"] == "sans") & (PAR_IMAGE["modele"] == "CNN_CBAM")]
normales_a_l_origine = cbam_sans[cbam_sans["condition"] == "originale"].groupby("chemin_relatif")["prediction"].apply(lambda s: int((s == 0).sum()))
pneumonie_si_sombre = cbam_sans[cbam_sans["condition"] == "luminosite_080"].groupby("chemin_relatif")["prediction"].apply(lambda s: int((s == 1).sum()))
bascules = np.array([i for i, chemin in enumerate(IDENTIFIANTS)
                     if REELS[i] == 0 and normales_a_l_origine.get(chemin, 0) >= 2 and pneumonie_si_sombre.get(chemin, 0) >= 2])
print(f"Radios NORMAL correctement classées à l'origine puis classées PNEUMONIA à × 0,80 par au moins 2 des 3 CBAM sans augmentation : {len(bascules)}")
if len(bascules) == 0:
    print("Aucune image ne remplit ce critère : figure non produite.")
else:
    choisies = bascules[:4]
    sombre_condition = next(c for c in CONDITIONS if c["id"] == "luminosite_080")
    sans_a, sans_b = paires_pneumonie("sans", "CNN_CBAM", choisies, originale_condition, sombre_condition)
    avec_a, avec_b = paires_pneumonie("avec", "CNN_CBAM", choisies, originale_condition, sombre_condition)
    titres = ["Image d'origine", "Image × 0,80", "CBAM sans augm.\ncarte pneumonie, origine", "CBAM sans augm.\ncarte pneumonie, × 0,80",
              "CBAM avec augm.\ncarte pneumonie, origine", "CBAM avec augm.\ncarte pneumonie, × 0,80"]
    fig, axes = plt.subplots(len(choisies), 6, figsize=(15, 2.7 * len(choisies)), squeeze=False)
    for ligne, i in enumerate(choisies):
        image = PREPAREES[i] / 255.0
        image_sombre = transformer(PREPAREES[i], MASQUES[i], sombre_condition, IDENTIFIANTS[i])
        afficher(axes[ligne, 0], image)
        afficher(axes[ligne, 1], image_sombre)
        afficher(axes[ligne, 2], image, sans_a[ligne])
        afficher(axes[ligne, 3], image_sombre, sans_b[ligne])
        afficher(axes[ligne, 4], image, avec_a[ligne])
        afficher(axes[ligne, 5], image_sombre, avec_b[ligne])
        axes[ligne, 0].set_ylabel(Path(IDENTIFIANTS[i]).name[:22], fontsize=8)
        if ligne == 0:
            for colonne, titre in enumerate(titres):
                axes[ligne, colonne].set_title(titre, fontsize=8)
    fig.suptitle("Radios NORMAL qui basculent vers PNEUMONIA une fois assombries (CBAM sans augmentation)\n"
                 "Pour chaque modèle, les deux cartes partagent la même échelle : une carte plus rouge = plus d'arguments pour PNEUMONIA.")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    enregistrer_figure(fig, SORTIE / "figures" / "bascules_assombrissement.png")
    plt.show()
    plt.close(fig)

ecrire_json({"id_protocole": ID_PROTOCOLE, "lignes_par_image": len(PAR_IMAGE), "bascules_cbam_sans_x080": int(len(bascules))},
            SORTIE / "termine.json")
print("Terminé. Fichiers à transmettre :")
for nom_fichier in ["diagnostic_separation_par_variable.csv", "diagnostic_modeles_simples.csv", "synthese_cartes_originale.csv",
                    "synthese_perturbations.csv", "comparaison_architectures_cartes.csv"]:
    print(" -", SORTIE / nom_fichier)
print("Figures :", SORTIE / "figures")

## Lecture des résultats et limites

**Diagnostic des raccourcis**

- Si le **format seul** sépare bien les classes (AUC de validation nettement au-dessus de 0,5, par exemple plus de 0,75), les deux classes ne proviennent pas des mêmes conditions d'acquisition. Le CNN peut exploiter ce format à travers l'épaisseur des bordures noires. C'est un risque de raccourci à discuter dans le mémoire.
- Si l'**intensité seule** sépare bien les classes, il faut regarder le sens : quand les radios PNEUMONIA sont en moyenne plus sombres, un modèle qui a appris « plus sombre = pneumonie » réagira exactement comme le CBAM sans augmentation face aux images assombries. L'intensité peut aussi refléter de vraies opacités ; ce diagnostic ne permet pas à lui seul de trancher.

**Cartes**

- Un **enrichissement du centre** nettement supérieur à 1 et peu de masse dans le padding indiquent que le modèle s'appuie surtout sur la région thoracique. Beaucoup de masse sur les bords ou dans le padding est un signal d'alerte (annotations, bordures, fond).
- Comparer le CBAM et le classique : le CBAM concentre-t-il davantage ses cartes au centre ? L'augmentation change-t-elle la répartition ?
- **Assombrissement** : pour les radios qui basculent de NORMAL à PNEUMONIA, une faible corrélation avec la carte d'origine et une hausse de la masse sur les bords ou dans le padding suggèrent un **raccourci**. Une forte corrélation, avec une carte qui reste au centre, suggère plutôt que le modèle **lit différemment la même zone** quand elle est plus sombre.

**Limites**

- Grad-CAM est une approximation locale, à la résolution de 28 × 28. Les cartes montrent où le modèle réagit, pas une preuve de cause.
- La zone « centre » est un rectangle fixe, pas une segmentation des poumons. Les proportions dépendent aussi du choix de la bande de 15 %, fixé avant l'analyse.
- La validation a déjà servi à choisir les modèles et les époques. Ces analyses sont descriptives et exploratoires.
- Les critères de choix des exemples sont fixés à l'avance : les trois premières images de chaque classe pour la figure 4, et pour la figure 5 les images qui basculent sous au moins deux graines. La figure 5 montre donc des cas choisis parce que la prédiction change : elle illustre un mécanisme, elle ne mesure pas sa fréquence.

**Références utiles pour la discussion :** Selvaraju et al. (2017), *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*, ICCV ; Woo et al. (2018), *CBAM: Convolutional Block Attention Module*, ECCV ; Geirhos et al. (2020), *Shortcut learning in deep neural networks*, Nature Machine Intelligence ; DeGrave et al. (2021), *AI for radiographic COVID-19 detection selects shortcuts over signal*, Nature Machine Intelligence.